In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision 
import cv2
from torchvision import transforms
from torchvision.transforms import v2
from torchvision.transforms import functional as TF
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

# Data Prepare

In [ ]:
train_dir = ''
all_labels = []
file_names = []
labels = []


for dirpath, dirnames, filenames in os.walk(train_dir):
    all_labels.extend(dirnames)
    for filename in filenames:
        file_names.append(filename)
        labels.append(dirpath.split('/')[-1])

all_labels = sorted(all_labels)

In [ ]:
def read_video(video_paths):
    frames = []

    cap = cv2.VideoCapture(video_paths)

    while True:
        ret, frame = cap.read()
        if ret == False:
            break

        frames.append(frame)

    return frames

In [ ]:
class VideoDataset(nn.Module):
    def __init__(self, root_dir, file_names, labels, transforms):
        self.root_dir = root_dir
        self.file_names = file_names
        self.labels = labels
        self.transforms = transforms

    def __len__(self):
        return len(self.file_names)

    def uniform_sampling(self, frames, target_frame):
        indices = np.linspace(0, len(frames) - 1, target_frame, dtype=np.int64)

        return frames[indices]

    def __getitem__(self, index):
        frames = read_video(os.path.join(self.root_dir, self.file_names[index]))
        frames = self.uniform_sampling(frames, 16)

        frames = self.transforms(frames)

        return frames, self.labels[index]

In [ ]:
simple_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale = True),
    v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

In [ ]:
train_filenames, val_filenames, train_labels, val_labels = train_test_split(file_names, labels, test_size=0.2, stratify=labels)

train_dataset = VideoDataset(train_dir, train_filenames, train_labels, simple_transforms)
val_dataset = VideoDataset(train_dir, val_filenames, val_labels, simple_transforms)

train_loader = DataLoader(train_dataset, batch_size = 8, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 8, shuff = False)

# Model Define

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

# Training

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)
criterion = nn.CrossEntropyLoss()

NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")

    # ============= Training ==================#
    model.train()
    train_loss = 0.0
    for vids, labels in tqdm(train_loader, desc = 'Train: '):
        optimizer.zero_grad()

        vids = vids.to(device)
        labels = labels.to(device)

        outputs = model(vids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)


    # ================ Valid ============== #
    model.eval()
    val_loss = 0.0
    with torch.inference_mode():
        for vids, labels in tqdm(val_loader, desc = 'Val: '):
            vids = vids.to(device)
            labels = labels.to(device)
    
            outputs = model(vids)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

        val_loss /= len(val_loader)

    print(f'Train loss: {train_loss} | Val loss: {val_loss}') 